# 11 · Evaluating RAG
**Concept:** measure retrieval (hit@k, MRR) with code and generation (faithfulness, relevance, correctness) with an LLM judge, on a golden set.
**TAC scenario:** QA scorecards for the system: did it find the right KB, and did the answer stick to it?

In [ ]:
from rag_common import MODEL, EMBED_MODEL, get_chat_model, get_vectorstore

model = get_chat_model()
vectorstore = get_vectorstore()          # opens the Chroma store built in Notebook 01
print(f"Chat: {MODEL} | Embeddings: {EMBED_MODEL} | chunks in store: {vectorstore._collection.count()}")

In [ ]:
import re
from collections import defaultdict
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever, ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMListwiseRerank
from rag_common import golden_set, load_chunks, doc_ids, hit, reciprocal_rank, format_docs, chroma_filter, bm25_tokenize

GOLDEN = golden_set()
answerable = [g for g in GOLDEN if g["type"] != "unanswerable"]
print(len(GOLDEN), "questions |", len(answerable), "answerable")

## 1. Pipelines to compare

In [ ]:
K = 4
chunks = load_chunks()
bm25 = BM25Retriever.from_documents(chunks, preprocess_func=bm25_tokenize, k=8)     # same tokenizer as Notebook 05
vector8 = vectorstore.as_retriever(search_kwargs={"k": 8})

def naive(g):      return vectorstore.similarity_search(g["question"], k=K)
def hybrid(g):     return EnsembleRetriever(retrievers=[bm25, vector8], weights=[0.5, 0.5]).invoke(g["question"])[:K]
def filtered(g):
    f = g["filters"]
    flt = chroma_filter(product=f.get("product"), version=f.get("version"))
    return vectorstore.similarity_search(g["question"], k=K, filter=flt) if flt else naive(g)
reranker = LLMListwiseRerank.from_llm(llm=model, top_n=K)
def hybrid_rerank(g):
    cands = EnsembleRetriever(retrievers=[bm25, vector8], weights=[0.5, 0.5]).invoke(g["question"])[:12]
    return reranker.compress_documents(cands, g["question"])

PIPELINES = {"naive": naive, "hybrid": hybrid, "filtered (oracle filters)": filtered, "hybrid + rerank": hybrid_rerank}

## 2. Retrieval metrics: hit@4 and MRR (pure code, no judge)

In [ ]:
results = {}
for name, fn in PIPELINES.items():
    rows = [(g, doc_ids(fn(g))) for g in answerable]
    results[name] = rows
    h = sum(hit(ids, g["expected_doc_ids"]) for g, ids in rows) / len(rows)
    mrr = sum(reciprocal_rank(ids, g["expected_doc_ids"]) for g, ids in rows) / len(rows)
    print(f"{name:26s} hit@{K} {h:5.0%}   MRR {mrr:.2f}")

In [ ]:
types = sorted({g["type"] for g in answerable})
print(f"{'hit@4 by type':26s}" + "".join(f"{t[:12]:>13s}" for t in types))
for name, rows in results.items():
    by = defaultdict(list)
    for g, ids in rows:
        by[g["type"]].append(hit(ids, g["expected_doc_ids"]))
    print(f"{name:26s}" + "".join(f"{sum(v)/len(v):>13.0%}" for t, v in sorted(by.items())))

## 3. Generation metrics with an LLM judge

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

gen = ChatPromptTemplate.from_messages([
    ("system", "Answer ONLY from the context and cite [doc_id]. If the context lacks the answer reply NOT_IN_KB."),
    ("human", "<context>\n{context}\n</context>\nQuestion: {question}"),
]) | model | StrOutputParser()

class Verdict(BaseModel):
    faithful: Literal["yes", "no"] = Field(description="Every claim is supported by the context")
    relevant: Literal["yes", "no"] = Field(description="The answer addresses the question")
    correct: Literal["yes", "no"] = Field(description="The answer agrees with the reference answer")
    reason: str

judge = model.with_structured_output(Verdict)

def evaluate_generation(fn, sample):
    scores, verdicts = defaultdict(int), []
    for g in sample:
        docs = fn(g)
        ans = gen.invoke({"context": format_docs(docs), "question": g["question"]})
        v = judge.invoke(f"QUESTION: {g['question']}\nCONTEXT:\n{format_docs(docs)}\nANSWER:\n{ans}\nREFERENCE ANSWER:\n{g['reference_answer']}")
        for k in ("faithful", "relevant", "correct"):
            scores[k] += v.__dict__[k] == "yes"
        verdicts.append((g, ans, v))
    return {k: v / len(sample) for k, v in scores.items()}, verdicts

sample = [g for g in GOLDEN if g["type"] in ("single_hop", "multi_hop", "error_code", "unanswerable")][::3]   # ~10 questions
gen_scores = {}
for name in ["naive", "hybrid + rerank"]:
    gen_scores[name], verdicts = evaluate_generation(PIPELINES[name], sample)
    print(f"{name:18s} " + "   ".join(f"{k} {v:.0%}" for k, v in gen_scores[name].items()))

## 4. Spot-check the judge (LLM judges have biases)

In [ ]:
for g, ans, v in verdicts[:3]:
    print(f"❓ {g['question']}\n💬 {ans[:250]}\n⚖️  faithful={v.faithful} relevant={v.relevant} correct={v.correct}: {v.reason}\n")

> Re-run this notebook whenever you change chunking, embeddings, prompts or the knowledge base. Treat the golden set like a regression suite.

---
**🔧 TAC takeaway:** QA scorecards for the system: retrieval metrics show whether it found the right KB; the judge shows whether the reply stuck to it. Start naive, measure, and keep only the fixes that earn their cost.